# 28. Hand-count validation month — May 2024 (the decisive test)

May 2024 has **recovered image contrast** (~39, essentially 2023-level) yet the model finds
**~0 worms/scene**. This resolves the open question:

- If you **see worms** here → the model is failing for a reason that is NOT contrast (scene/framing/
  appearance change) → the post-Sep-2023 collapse is a model artifact, retraining justified.
- If you see **few / no worms** → part of the collapse is a **real biological decline**, not just the model.

Same workflow as before: set `IDX`, look, set `MY_COUNT`, Shift+Enter. No CSV editing.

In [1]:
import csv
from pathlib import Path

# May-2024 frames + blank CSV were built by scripts/setup_handcount_month.py 2024/05
FRAMES_DIR = Path('./handcount_2024_05_frames')
HANDCOUNT_CSV = Path('./handcount_2024_05.csv')
frames = sorted(FRAMES_DIR.glob('*.png'))
print(f'{len(frames)} frames ready; CSV = {HANDCOUNT_CSV}')

15 frames ready; CSV = handcount_2024_05.csv


In [17]:
# ── Hand-count viewer + recorder — ONE frame at a time ──
#   1. Set IDX to the frame you want. Leave MY_COUNT = None to just LOOK.
#   2. Count the worms, set MY_COUNT = <your number>, Shift+Enter: it saves to the CSV.
#   3. Bump IDX, repeat. Re-running a frame with a new number overwrites it (fix mistakes freely).
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

IDX = 15            # <-- which frame: 0 .. 14
MY_COUNT = 7    # <-- None = view only;  a number = record THIS frame's worm count

f = frames[IDX]
stem = f.stem
fig, ax = plt.subplots(figsize=(15, 8))
ax.imshow(mpimg.imread(f))
ax.set_axis_off()
ax.set_title(f'[{IDX + 1}/{len(frames)}]  {stem}', fontsize=12)
plt.show()

with HANDCOUNT_CSV.open() as fh:
    reader = csv.DictReader(fh)
    fields = list(reader.fieldnames)
    rows = list(reader)

if MY_COUNT is not None:
    matched = False
    for r in rows:
        if r['stem'] == stem:
            r['human_count'] = MY_COUNT
            matched = True
    if matched:
        with HANDCOUNT_CSV.open('w', newline='') as fh:
            w = csv.DictWriter(fh, fieldnames=fields)
            w.writeheader()
            w.writerows(rows)
        print(f'saved  human_count = {MY_COUNT}  for {stem}')
    else:
        print(f'!! no CSV row matches {stem} — nothing written')

filled = sum(1 for r in rows if str(r.get('human_count', '')).strip())
print(f'progress: {filled}/{len(frames)} frames counted')
if MY_COUNT is not None and IDX + 1 < len(frames):
    print(f'-> next: set IDX = {IDX + 1}, MY_COUNT = <count for that frame>')

IndexError: list index out of range